In [ ]:
# -*- coding: utf-8 -*-

import sys
import argparse
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.func import functional_call, vmap, jacrev

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")   # TF32 for all fp32 matmuls


def scaled_hparams(L, L0=10, n_chains0=600, sweeps0=8, burnin0=10):
    """Scale sampler hyperparameters with lattice size, matching how much
    Monte Carlo effort SR actually needs as the problem grows."""
    scale = L / L0
    n_chains        = int(round(n_chains0))               # linear in L (memory-safe)
    sweeps_per_iter = max(sweeps0, int(round(sweeps0)))    # decorrelation ~ L
    burn_in_sweeps  = max(burnin0, int(round(burnin0)))
    return n_chains, sweeps_per_iter, burn_in_sweeps


# ==========================================================================
# 1. Model: 2D ResNet-CNN neural quantum state
# ==========================================================================

class ResBlock2D(nn.Module):
    """Pre-activation residual block with circular (periodic-BC) convolutions."""

    def __init__(self, channels: int = 8, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.norm  = nn.GroupNorm(1, channels)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, padding=pad, padding_mode="circular")
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, padding=pad, padding_mode="circular")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.gelu(self.norm(x))
        h = F.gelu(self.conv1(h))
        h = self.conv2(h)
        return x + h


def complex_logsumexp(z: torch.Tensor, dim: int) -> torch.Tensor:
    """Numerically stable log(sum(exp(z))) for complex z."""
    m = torch.amax(z.real, dim=dim, keepdim=True)
    m = torch.where(torch.isfinite(m), m, torch.zeros_like(m))
    return torch.log(torch.sum(torch.exp(z - m), dim=dim)) + m.squeeze(dim)


class NQS_CNN_2D(nn.Module):
    """
    2D ResNet-CNN NQS ansatz (Purely Real).
    Input : s ∈ {±1}^{L×L},  shape (batch, L, L)
    Output: ln Ψ_θ(s),       shape (batch,), float32
    """

    def __init__(self, channels: int = 8, depth: int = 4, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.embed      = nn.Conv2d(1, channels, kernel_size, padding=pad, padding_mode="circular")
        self.blocks     = nn.ModuleList([ResBlock2D(channels, kernel_size) for _ in range(depth)])
        self.final_norm = nn.GroupNorm(1, channels)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        x = self.embed(s.unsqueeze(1).float())          # (B, C, L, L)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)

        # Flatten all features and pool them into a single real number per sample
        return torch.logsumexp(x.flatten(1), dim=1)     # (B,) real


# ==========================================================================
# 2. Monte Carlo sampler: batched Metropolis with single spin-flip proposals
# ==========================================================================

class MetropolisSampler:
    """
    n_chains independent Metropolis chains.  All chains are updated in one
    batched forward pass per step — the parallelism is across chains, not time.
    """

    def __init__(self, model: NQS_CNN_2D, L: int, n_chains: int = 128, device: str = "cpu"):
        # CHANGE: model wrapped in torch.compile for this sampler's private use only.
        # The uncompiled `model` reference used elsewhere (Jacobian, local energy)
        # is untouched — same parameters, no drift, just a different call path.
        self.model    = torch.compile(model, mode="reduce-overhead")
        self.L        = L
        self.n_chains = n_chains
        self.device   = device
        self._bn      = torch.arange(n_chains, device=device)   # cached batch index

        self.state = torch.where(
            torch.rand(n_chains, L, L, device=device) < 0.5,
            torch.full((), -1.0, device=device),
            torch.full((), +1.0, device=device),
        )
        with torch.no_grad():
            # CHANGE: .clone() required — under CUDA graphs, the model's output
            # aliases a reused static buffer that gets overwritten on the next call.
            self.lnpsi = self.model(self.state).clone()

    @torch.no_grad()
    def sweep(self, n_sweeps: int = 1) -> torch.Tensor:
        L, n = self.L, self.n_chains
        total = n_sweeps * L * L
        bn    = self._bn

        ri = torch.randint(0, L, (total, n), device=self.device)
        rj = torch.randint(0, L, (total, n), device=self.device)
        ru = torch.rand(total, n, device=self.device)

        for t in range(total):
          i, j = ri[t], rj[t]

          # unconditional proposal flip — fixed shape, leave this alone
          self.state[bn, i, j] *= -1

          lnpsi_new = self.model(self.state).clone()
          accept = ru[t].log() < 2.0 * (lnpsi_new.real - self.lnpsi.real)

          # shape-stable revert — replaces the old boolean-mask revert
          current = self.state[bn, i, j]
          sign = torch.where(accept, 1.0, -1.0)
          self.state[bn, i, j] = current * sign

          self.lnpsi = torch.where(accept, lnpsi_new, self.lnpsi)

        return self.state


# ==========================================================================
# 3. Hamiltonian: vectorised local energy for 2D TFIM
# ==========================================================================

def local_energy_tfim(
    model      : NQS_CNN_2D,
    s          : torch.Tensor,
    J          : float = 1.0,
    h          : float = 1.0,
    lnpsi_s    : torch.Tensor = None,
    chunk_size : int   = 8192,
) -> torch.Tensor:
    """
    E_loc(s) = -J Σ_{<ij>} s_i s_j  -  h Σ_i [Ψ(s^i)/Ψ(s)]
    """
    B, L, _ = s.shape
    device   = s.device
    N        = L * L

    diag = -J * (
        (s * torch.roll(s, -1, 1)).sum((1, 2)) +
        (s * torch.roll(s, -1, 2)).sum((1, 2))
    )

    if lnpsi_s is None:
        lnpsi_s = model(s)   # (B,) complex

    flip_i = torch.arange(L, device=device).repeat_interleave(L)
    flip_j = torch.arange(L, device=device).repeat(L)

    s_all  = s.unsqueeze(1).expand(B, N, L, L).clone().reshape(B * N, L, L)
    bn_idx = torch.arange(B * N, device=device)
    s_all[bn_idx, flip_i.repeat(B), flip_j.repeat(B)] *= -1

    lnpsi_flip = torch.cat(
        [model(chunk) for chunk in s_all.split(chunk_size)]
    ).reshape(B, N)

    offdiag = -h * torch.exp(lnpsi_flip - lnpsi_s.unsqueeze(1)).sum(1)
    return diag.to(torch.complex64) + offdiag


# ==========================================================================
# 4. SPRING momentum — dual/kernel formulation
# ==========================================================================

def compute_lnpsi_jacobian(model: NQS_CNN_2D, s: torch.Tensor) -> torch.Tensor:
    params = {k: v.detach() for k, v in model.named_parameters()}
    param_names = list(params.keys())

    def lnpsi_fn(params_dict, s_single):
        return functional_call(model, params_dict, (s_single.unsqueeze(0),))[0]

    jac = vmap(jacrev(lnpsi_fn), in_dims=(None, 0))(params, s)

    flat_chunks = []
    for name in param_names:
        j = jac[name] # (n_chains, *param_shape)
        flat_chunks.append(j.reshape(s.shape[0], -1))

    return torch.cat(flat_chunks, dim=1) # (n_chains, n_params) float32


def sr_spring_update(
    O          : torch.Tensor,
    e_loc      : torch.Tensor,
    p_prev     : torch.Tensor,
    mu         : float = 0.9,
    diag_shift : float = 1e-3,
) -> torch.Tensor:
    n = O.shape[0]
    Obar = O - O.mean(0, keepdim=True)
    ebar = (e_loc - e_loc.mean()).to(O.dtype)

    correction = Obar @ p_prev.to(O.dtype)
    eps_tilde  = ebar - mu * correction

    # Promote Obar and eps_tilde to FP64 BEFORE matrix multiplication to avoid TF32 noise
    dtype_64 = torch.complex128 if O.is_complex() else torch.float64
    Obar_64 = Obar.to(dtype_64)
    eps_64  = eps_tilde.to(dtype_64)

    # Compute T in FP64
    T = Obar_64 @ Obar_64.conj().T

    # Explicitly enforce symmetry/Hermiticity to guarantee real non-negative eigenvalues
    T = 0.5 * (T + T.conj().T)

    # Add diagonal shift
    diag_indices = torch.arange(n, device=T.device)
    T[diag_indices, diag_indices] += diag_shift

    # Solve linear system in double precision
    y_64 = torch.linalg.solve(T, eps_64)
    x_64 = (Obar_64.conj().T @ y_64).real
    x = x_64.to(p_prev.dtype)

    return mu * p_prev + x


# ==========================================================================
# 5. Training loop: VMC with SR+SPRING momentum
# ==========================================================================

def train(
    L               : int   = 10,
    depth           : int   = 4,
    channels        : int   = 8,
    n_chains        : int   = 128,
    n_iters         : int   = 1000,
    burn_in_sweeps  : int   = 5,
    sweeps_per_iter : int   = 2,
    J               : float = 1.0,
    h               : float = 1.0,
    device          : str   = "cpu",
    chunk_size      : int   = 2048,
    max_grad_norm   : float = 1.0,
    sr_lr           : float = 0.02,
    sr_diag_shift_start : float = 1e-2,
    sr_diag_shift_end   : float = 1e-4,
    sr_momentum     : float = 0.9,
    diag_interval   : int   = 0,     # CHANGE: 0 = diagnostics off (default). N>0 = every N iters.
    time_breakdown  : bool  = True, # CHANGE: optional per-iteration timing breakdown
):
    model = NQS_CNN_2D(channels=channels, depth=depth).to(device)
    sampler = MetropolisSampler(model, L, n_chains=n_chains, device=device)
    sampler.sweep(burn_in_sweeps)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"[info] optimizer = spring   n_params = {n_params}   n_chains = {n_chains}")
    if diag_interval > 0:
        print(f"[info] full FP64 eigendecomposition diagnostics ON, every {diag_interval} iters")
    else:
        print("[info] full FP64 eigendecomposition diagnostics OFF (diag_interval=0)")

    p_prev = torch.zeros(n_params, device=device)
    energy_history = []

    # CHANGE: timing accumulators, only used if time_breakdown=True
    timings = {"sweep": 0.0, "jacobian": 0.0, "linalg": 0.0}
    warmup_iters = 5
    is_cuda = (device == "cuda") or (isinstance(device, torch.device) and device.type == "cuda")

    # last computed diagnostic values, kept across iterations for logging continuity
    lam_min = lam_max = cond_num = None
    eff_rank = None

    # CHANGE: iters_completed tracks how far we actually got, so timing stats
    # and the "stopped early" message are correct even if we're interrupted.
    iters_completed = 0
 
    # CHANGE: catching KeyboardInterrupt here means a Ctrl-C (or Colab's
    # "Interrupt execution") stops the loop cleanly but still falls through
    # to the timing summary and `return model, energy_history` below —
    # so the caller always gets back a usable model + plot, partial or not.
    try:
        for it in range(n_iters):
            iters_completed = it + 1
 
            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
 
            sampler.sweep(sweeps_per_iter)
 
            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t1 = time.perf_counter()
 
            s = sampler.state.clone()
 
            with torch.no_grad():
                e_loc = local_energy_tfim(
                    model, s, J=J, h=h,
                    lnpsi_s=sampler.lnpsi.clone(),
                    chunk_size=chunk_size,
                )
                e_mean = e_loc.mean()
                e_var = e_loc.real.var().item() # Diagnostic: track variance
 
            frac = it / max(n_iters - 1, 1)
            diag_shift = sr_diag_shift_start + frac * (sr_diag_shift_end - sr_diag_shift_start)
 
            run_full_diag = diag_interval > 0 and (it % diag_interval == 0)
 
            with torch.no_grad():
                O = compute_lnpsi_jacobian(model, s)
 
                if time_breakdown and is_cuda:
                    torch.cuda.synchronize()
                t2 = time.perf_counter()
 
                # ------ NUMERICALLY STABLE DIAGNOSTICS (gated) ------
                if run_full_diag:
                    Obar = O - O.mean(0, keepdim=True)
                    dtype_64 = torch.complex128 if O.is_complex() else torch.float64
                    Obar_64 = Obar.to(dtype_64)
 
                    T_fp64 = Obar_64 @ Obar_64.conj().T
                    T_fp64 = 0.5 * (T_fp64 + T_fp64.conj().T)  # enforce Hermiticity
 
                    # full FP64 eigendecomposition — O(n_chains^3), diagnostics only
                    eigvals = torch.linalg.eigvalsh(T_fp64).real
                    lam_min = eigvals.min().item()
                    lam_max = eigvals.max().item()
                    cond_num = lam_max / max(lam_min, 1e-15)
                    eff_rank = (eigvals > 1e-6).sum().item()
                # ------------------------------------------------------
 
                max_param_norm = torch.nn.utils.parameters_to_vector(model.parameters()).norm().item()
 
                # NOTE: sr_spring_update's own T + solve is load-bearing and always runs,
                # regardless of diag_interval — this is not part of the gated diagnostic.
                p_prev = sr_spring_update(O, e_loc, p_prev, mu=sr_momentum, diag_shift=diag_shift)
                step = p_prev
 
                step_norm = step.norm()
                applied_step = step * (max_grad_norm / step_norm) if step_norm > max_grad_norm else step
 
                flat = torch.nn.utils.parameters_to_vector(model.parameters())
                flat = flat - sr_lr * applied_step.to(flat.dtype)
                torch.nn.utils.vector_to_parameters(flat, model.parameters())
 
            if time_breakdown and is_cuda:
                torch.cuda.synchronize()
            t3 = time.perf_counter()
 
            if time_breakdown and it >= warmup_iters:
                timings["sweep"]    += (t1 - t0)
                timings["jacobian"] += (t2 - t1)
                timings["linalg"]   += (t3 - t2)
 
            e_site = e_mean.real.item() / (L * L)
            energy_history.append(e_site)
 
            if it % 10 == 0:
                if run_full_diag:
                    print(f"iter {it:4d} | E/site = {e_site:.12f} | Var = {e_var:.5f} | "
                          f"Rank = {eff_rank}/{n_chains} | min_eig = {lam_min:.2e} | "
                          f"Cond = {cond_num:.1e} | |w| = {max_param_norm:.2f}")
                else:
                    print(f"iter {it:4d} | E/site = {e_site:.12f} | Var = {e_var:.5f} | "
                          f"|w| = {max_param_norm:.2f}  (full-diag off)")
 
    except KeyboardInterrupt:
        print(f"\n[info] KeyboardInterrupt caught after {iters_completed}/{n_iters} iterations — "
              f"stopping early. Returning the partially-trained model and energy history so far.")
 
    # This runs whether training finished normally or was interrupted above,
    # so you always get a timing summary (over whatever iterations actually ran)
    # and a usable (model, energy_history) to plot.
    if time_breakdown:
        n_timed = max(iters_completed - warmup_iters, 1)
        total = sum(timings.values())
        print("\n--- Per-iteration breakdown (excl. warmup, sync'd) ---")
        for k, v in timings.items():
            pct = (v / total * 100) if total > 0 else 0.0
            print(f"{k:>10}: {v/n_timed*1000:.1f} ms/iter  ({pct:.1f}%)")
 
    return model, energy_history


# ==========================================================================
# 6. Entry point — compatible with both Colab and CLI
# ==========================================================================

p = argparse.ArgumentParser(description="Train a 2D CNN NQS on the TFIM.")
p.add_argument("--L",               type=int,   default=16,  help="lattice side length")
p.add_argument("--depth",           type=int,   default=9,   help="number of residual blocks")
p.add_argument("--channels",        type=int,   default=8,   help="conv channels (must be even)")
p.add_argument("--n_iters",         type=int,   default=1200, help="optimizer steps")
p.add_argument("--J",               type=float, default=1.0,  help="Ising coupling")
p.add_argument("--h",               type=float, default=3.05,  help="transverse field")
p.add_argument("--chunk_size",      type=int,   default=8192 * 16, help="flip-batch chunk size (tune for VRAM)")
p.add_argument("--device",          type=str,   default="cuda" if torch.cuda.is_available() else "cpu")
p.add_argument("--sr_lr",           type=float, default=0.02, help="SPRING step size")
p.add_argument("--sr_diag_shift_start", type=float, default=1e-2)
p.add_argument("--sr_diag_shift_end",   type=float, default=1e-4)
p.add_argument("--sr_momentum",     type=float, default=0.9, help="SPRING momentum factor mu")
# CHANGE: new CLI flags for diagnostics/timing, both off by default
p.add_argument("--diag_interval",   type=int,   default=100,
               help="run full FP64 eigendecomposition diagnostics every N iters (0 = never, default)")
p.add_argument("--time_breakdown",  action="store_true",
               help="print per-iteration timing breakdown (sweep/jacobian/linalg)")

is_colab = "google.colab" in sys.modules
args = p.parse_args([] if is_colab else None)

n_chains, sweeps_per_iter, burn_in_sweeps = scaled_hparams(args.L)
print(f"[info] scaled for L={args.L}: n_chains={n_chains}, "
      f"sweeps_per_iter={sweeps_per_iter}, burn_in_sweeps={burn_in_sweeps}")

energy_history = []
model = None
try:
    model, energy_history = train(
        L=args.L, depth=args.depth, channels=args.channels,
        n_chains=n_chains, n_iters=args.n_iters,
        sweeps_per_iter=sweeps_per_iter, burn_in_sweeps=burn_in_sweeps,
        J=args.J, h=args.h,
        device=args.device, chunk_size=args.chunk_size,
        sr_lr=args.sr_lr,
        sr_diag_shift_start=args.sr_diag_shift_start,
        sr_diag_shift_end=args.sr_diag_shift_end,
        sr_momentum=args.sr_momentum,
        diag_interval=args.diag_interval,
        time_breakdown=True,
    )
except KeyboardInterrupt:
    print("\n[info] KeyboardInterrupt caught outside the training loop — "
          "plotting whatever energy history was recorded so far.")
 
if len(energy_history) > 0:
    print(f"\n[info] Final E/site = {energy_history[-1]:.6f}   "
          f"(over {len(energy_history)} recorded iterations)")
 
    plt.figure(figsize=(6, 4))
    plt.plot(energy_history)
    plt.xlabel("iteration")
    plt.ylabel("E / site")
    title_suffix = "" if len(energy_history) >= args.n_iters else "  [partial / interrupted run]"
    plt.title(f"2D CNN NQS (SPRING) — {args.L}×{args.L} TFIM  (J={args.J}, h={args.h}){title_suffix}")
    plt.tight_layout()
    plt.show()
else:
    print("\n[info] No iterations completed — nothing to plot.")